# Project Journal 03: Multi-Vintage Field Review and Stronger Classical Baseline

This entry records the point where the project stopped asking only whether constrained field outputs looked plausible and started comparing them against a stronger field-aware classical baseline.


## Context

Journal 02 established a usable real-data workflow for the `p07` Sleipner line family, but the project still needed a stronger baseline to avoid an overly flattering comparison.

This stage focused on two questions:

- does a field-aware cross-equalized difference baseline explain the field behavior just as well as the hybrid method?
- do the constrained multi-vintage results remain temporally plausible when judged against that stronger baseline?

This is the stage where the project moved from "we can run field data" to "we can argue that the comparison is not trivial or unfair."


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

ROOT = Path.cwd().resolve()
if not (ROOT / 'runs').exists():
    ROOT = ROOT.parent.resolve()

def load_json(path):
    with Path(path).open('r', encoding='utf-8') as handle:
        return json.load(handle)

def show_images(image_paths, titles, figsize=(18, 8)):
    fig, axes = plt.subplots(1, len(image_paths), figsize=figsize)
    if len(image_paths) == 1:
        axes = [axes]
    for ax, image_path, title in zip(axes, image_paths, titles):
        image = mpimg.imread(image_path)
        ax.imshow(image)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

ROOT

p07_summary = load_json(ROOT / 'runs' / 'sleipner_manifest' / 'results' / 'summary.json')
field = p07_summary['Field']
temporal = field['temporal_consistency']
field


## What Changed This Stage

The main additions in this stage were:

- a stronger classical field baseline based on cross-equalized difference scoring
- shared thresholding across field pairs instead of per-pair threshold tuning
- a multi-vintage review over `2001`, `2004`, and `2006`
- temporal consistency metrics based on support overlap and lateral trace footprint

This matters because industrial 4D monitoring is heavily shaped by repeatability and cross-equalization, so a fair paper cannot compare the hybrid model only against weak raw-difference baselines.


In [ ]:
rows = []
for pair_name, metrics in field['pairs'].items():
    rows.append({
        'pair': pair_name,
        'cross_eq_trace_iou_2010': metrics['cross_equalized_difference']['trace_iou_with_2010_support'],
        'cross_eq_trace_fraction': metrics['cross_equalized_difference']['predicted_trace_fraction'],
        'hybrid_constrained_trace_iou_2010': metrics['hybrid_ml_constrained']['trace_iou_with_2010_support'],
        'hybrid_constrained_trace_fraction': metrics['hybrid_ml_constrained']['predicted_trace_fraction'],
        'hybrid_constrained_trace_outside_support': metrics['hybrid_ml_constrained']['trace_fraction_outside_2010_support'],
    })

comparison = pd.DataFrame(rows)
display(comparison)

progress = pd.DataFrame({
    'pair': temporal['ordered_pairs'],
    'trace_fraction': [temporal['constrained_trace_fraction_by_pair'][name] for name in temporal['ordered_pairs']],
    'support_iou_2010': [temporal['constrained_support_iou_by_pair'][name] for name in temporal['ordered_pairs']],
})
display(progress)


## Current Results

This stage produced a stronger and more defensible field comparison.

What held up:

- the cross-equalized baseline spread over almost every support trace, so it was not selective enough to explain the constrained hybrid result away
- the constrained hybrid stayed much tighter laterally than the cross-equalized baseline
- support overlap grew from `2001` to `2006`, which is physically more believable than the earlier pair-specific threshold behavior

What still needed caution:

- the raw hybrid model still leaked badly outside the storage interval before masking
- the constrained result depended on benchmark structure and shared thresholds
- the pixel-area progression was still weaker than the support-footprint progression

So the honest field takeaway from this stage was: the constrained hybrid had a better lateral support story than the stronger classical baseline, but it was still not an unconstrained field-segmentation success.


In [ ]:
show_images(
    [
        ROOT / 'runs' / 'sleipner_manifest' / 'results' / 'figures' / 'field_sleipner_2001_inline_1840_mid_cross_equalized_difference.png',
        ROOT / 'runs' / 'sleipner_manifest' / 'results' / 'figures' / 'field_sleipner_2001_inline_1840_mid_hybrid_example.png',
        ROOT / 'runs' / 'sleipner_manifest' / 'results' / 'figures' / 'field_sleipner_2001_inline_1840_mid_hybrid_constrained.png',
    ],
    ['2001 cross-equalized', '2001 raw hybrid', '2001 constrained hybrid'],
    figsize=(18, 6),
)

show_images(
    [
        ROOT / 'runs' / 'sleipner_manifest' / 'results' / 'figures' / 'field_sleipner_2006_inline_1840_mid_cross_equalized_difference.png',
        ROOT / 'runs' / 'sleipner_manifest' / 'results' / 'figures' / 'field_sleipner_2006_inline_1840_mid_hybrid_example.png',
        ROOT / 'runs' / 'sleipner_manifest' / 'results' / 'figures' / 'field_sleipner_2006_inline_1840_mid_hybrid_constrained.png',
    ],
    ['2006 cross-equalized', '2006 raw hybrid', '2006 constrained hybrid'],
    figsize=(18, 6),
)


## Open Problems

The field review still had some real limitations:

- the public 2010 plume-support product is a structural envelope, not a dense pixel-level label for each earlier vintage
- zero constrained leakage outside the reservoir comes from the benchmark mask, so it should not be presented as unconstrained model skill
- the multi-vintage evidence was still indirect because it did not include a same-year direct 2010 benchmark check on a matching processing family

That is why the next stage needed a direct `94p10 -> 10p10` audit rather than more synthetic experimentation.


In [ ]:
print(field['support_note'])
print('Area deltas:', temporal['constrained_area_deltas'])
print('Trace-fraction deltas:', temporal['constrained_trace_fraction_deltas'])
print('Pairwise IoU:', temporal['constrained_pairwise_iou'])
print('Area non-decreasing:', temporal['constrained_area_non_decreasing'])
print('Trace fraction non-decreasing:', temporal['constrained_trace_fraction_non_decreasing'])
print('Support IoU non-decreasing:', temporal['constrained_support_iou_non_decreasing'])


## Next Stage

The next stage had to answer a sharper question:

- if the project is really learning something useful, does the constrained hybrid still look strong on a direct same-year 2010 benchmark audit?

The immediate tasks were:

1. export the `94p10` and `10p10` inline sections
2. build the benchmark mask and 2010 plume-support envelope on that exact geometry
3. rerun field evaluation while reusing the same trained artifacts, with the provenance saved explicitly
4. compare the constrained hybrid directly against the cross-equalized baseline on the 2010 benchmark check
